# G05: MLPs as Knowledge Stores

**Prerequisites:** G02 (loading GPT-2, cache basics).

**What you'll learn:**
- How MLP layers work inside a transformer
- Neuron→logit analysis: what does each neuron promote in the output?
- What activates a neuron: reading the "key" (W_in rows)
- Layer ablation: which layers store factual knowledge?
- Dead neurons and the capacity problem
- Polysemanticity: why neurons respond to multiple unrelated things

Attention heads route information. MLPs transform it. This tutorial teaches you to understand what transformations GPT-2's MLPs perform and what knowledge they store.

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.neurons import (
    neuron_to_logit, neuron_attribution, top_activating_tokens,
    dead_neuron_fraction, neuron_logit_effects, get_neuron_activations,
)

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"GPT-2 Small: {model.cfg.n_layers} layers, d_mlp={model.cfg.d_mlp}")

## How an MLP Layer Works

Each transformer block has an MLP that processes every token position independently. The MLP is a two-layer feedforward network:

```
MLP(x) = GELU(x @ W_in + b_in) @ W_out + b_out
```

First, `W_in` projects from `d_model` (768) up to `d_mlp` (3072) — a 4× expansion. The GELU activation function introduces nonlinearity and sparsity: many neurons end up near zero. Then `W_out` projects back down to `d_model`. Each of the 3072 "neurons" is one dimension in this hidden layer.

The key insight from [Geva et al. 2021](https://arxiv.org/abs/2012.14913): each neuron has a **key** and a **value**. The `W_in` row for neuron *i* is the key — it determines WHEN the neuron fires (what input patterns activate it). The `W_out` column for neuron *i* is the value — it determines WHAT the neuron writes to the residual stream. This is why MLPs have been called **key-value memories**.

## Neuron→Logit Analysis

We can trace what each neuron promotes in the output vocabulary by projecting its `W_out` column through the unembedding matrix `W_U`. When neuron *i* fires, it writes `W_out[i, :]` into the residual stream. That vector eventually gets multiplied by `W_U` to produce logits. So `W_out[i, :] @ W_U` gives us the neuron's direct effect on every token's logit. This reveals the neuron's "semantic role" — what it's trying to say.

In [ ]:
# Run on a factual prompt and find which neurons fire at the prediction position
text = "The capital of France is"
tokens = model.to_tokens(text)
token_labels = [tokenizer.decode([t]) for t in np.array(tokens)]

logits, cache = model.run_with_cache(tokens)
pred_token = int(jnp.argmax(logits[-1]))
print(f"Input: {text!r}")
print(f"Tokens: {token_labels}")
print(f"Model predicts: {tokenizer.decode([pred_token])!r}")
print()

# Get the Paris token ID for later analysis
paris_token = tokenizer.encode(" Paris")[0]
paris_prob = float(jax.nn.softmax(logits[-1])[paris_token])
print(f"P(' Paris') = {paris_prob:.4f}")
print()

# Find the most active neurons at the last position across several layers
print("Top-activated neurons at the last position:")
print("=" * 60)
interesting_neurons = []  # (layer, neuron, activation) for later analysis

for layer in [6, 8, 9, 10, 11]:
    acts = get_neuron_activations(cache, layer)  # [seq_len, d_mlp]
    last_pos_acts = acts[-1]  # [d_mlp]
    top_5 = np.argsort(np.abs(last_pos_acts))[::-1][:5]

    print(f"\nLayer {layer}:")
    for neuron in top_5:
        act = last_pos_acts[neuron]
        promoted, suppressed = neuron_to_logit(model, layer, int(neuron), k=5)
        top_words = [f"{tokenizer.decode([tid])!r}" for tid, _ in promoted[:3]]
        print(f"  Neuron {neuron:4d} (act={act:+.2f}) promotes: {', '.join(top_words)}")
        if abs(act) > 1.0:
            interesting_neurons.append((layer, int(neuron), float(act)))

print(f"\nFound {len(interesting_neurons)} strongly-active neurons for deeper analysis.")

In [ ]:
# Neuron attribution: which neurons contribute most to the " Paris" prediction?
print(f"Neuron attribution for predicting ' Paris' (token {paris_token})")
print("=" * 65)

# Check each layer's attribution
layer_top_neurons = {}  # layer -> top (neuron, attribution) pairs

for layer in range(model.cfg.n_layers):
    attr = neuron_attribution(model, cache, layer, token=paris_token, pos=-1)
    top_k = np.argsort(np.abs(attr))[::-1][:3]
    total_attr = float(np.sum(attr))
    layer_top_neurons[layer] = [(int(n), float(attr[n])) for n in top_k]

    if abs(total_attr) > 0.1:
        neuron_strs = [f"N{n}({attr[n]:+.3f})" for n in top_k]
        print(f"  Layer {layer:2d}: total={total_attr:+.3f}  top neurons: {', '.join(neuron_strs)}")

# Plot total attribution by layer
layer_totals = []
for layer in range(model.cfg.n_layers):
    attr = neuron_attribution(model, cache, layer, token=paris_token, pos=-1)
    layer_totals.append(float(np.sum(attr)))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in layer_totals]
ax.bar(range(model.cfg.n_layers), layer_totals, color=colors)
ax.set_xlabel("Layer")
ax.set_ylabel("Total MLP Attribution")
ax.set_title("MLP Neuron Attribution for ' Paris' by Layer\n(Red = promotes, Blue = suppresses)")
ax.set_xticks(range(model.cfg.n_layers))
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## What Activates a Neuron?

The `W_in` row for each neuron acts as a "detector" or "key." The residual stream at each position is dotted with this key vector. When the dot product is high, the neuron fires strongly. We can find what activates a neuron empirically by running diverse text through the model and recording where each neuron fires most. This reveals the neuron's "trigger" — the input patterns it responds to.

In [ ]:
# Prepare diverse text sequences to probe neuron activations
test_texts = [
    "The capital of France is Paris and the capital of Germany is Berlin",
    "The president of the United States lives in Washington DC",
    "London is a city in England and Tokyo is a city in Japan",
    "The cat sat on the mat and the dog chased the ball",
    "She walked to the store and bought some milk and bread",
    "The stock market dropped by five percent on Monday morning",
    "In 1969 the first humans landed on the moon during the Apollo",
    "Python is a programming language used for machine learning and data",
    "The recipe calls for two cups of flour and one cup of sugar",
    "Einstein published his theory of general relativity in nineteen fifteen",
    "The river flows through the valley and into the ocean near the coast",
    "Mozart composed his first symphony at the age of eight years old",
]
test_sequences = [model.to_tokens(t) for t in test_texts]

# Find what activates the most important neurons from our attribution analysis
# Pick neurons with the highest absolute attribution for " Paris"
if interesting_neurons:
    # Sort by absolute activation, take top 3
    top_interesting = sorted(interesting_neurons, key=lambda x: abs(x[2]), reverse=True)[:3]
else:
    # Fallback: use layer 9, neuron with highest activation
    acts = get_neuron_activations(cache, 9)
    top_n = int(np.argmax(np.abs(acts[-1])))
    top_interesting = [(9, top_n, float(acts[-1, top_n]))]

print("What activates our key neurons?")
print("=" * 65)

for layer, neuron, orig_act in top_interesting:
    results = top_activating_tokens(model, test_sequences, layer, neuron, k=8)

    print(f"\nLayer {layer}, Neuron {neuron} (activation on 'France' prompt: {orig_act:+.2f})")
    print(f"  Top activating tokens across diverse text:")
    for prompt_idx, pos, activation in results:
        tok_id = int(test_sequences[prompt_idx][pos])
        tok_str = tokenizer.decode([tok_id])
        # Show context: a few tokens around the activating position
        seq = test_sequences[prompt_idx]
        start = max(0, pos - 2)
        end = min(len(seq), pos + 3)
        context = tokenizer.decode(np.array(seq[start:end]))
        print(f"    act={activation:+.2f}  token={tok_str!r:12s}  context: ...{context}...")

## Layer Ablation: Which MLPs Store Knowledge?

By zeroing out each MLP layer's output and measuring how the prediction changes, we can see which layers are critical. For factual recall ("The capital of France is" → " Paris"), we expect specific layers to matter a lot. For syntactic predictions ("The cat sat on the" → " mat"), the knowledge is often more distributed across layers.

In [ ]:
# MLP layer ablation experiment
# Compare factual recall vs syntactic prediction

def ablate_mlp_layers(text, target_word):
    """Zero out each MLP layer and measure P(target) at last position."""
    tokens = model.to_tokens(text)
    target_token = tokenizer.encode(target_word)[0]

    # Baseline
    logits_base = model(tokens)
    probs_base = jax.nn.softmax(logits_base[-1])
    base_prob = float(probs_base[target_token])

    # Ablate each layer
    probs_ablated = []
    for layer in range(model.cfg.n_layers):
        hook_name = f"blocks.{layer}.hook_mlp_out"
        hooks = [(hook_name, lambda x, name: jnp.zeros_like(x))]
        logits_abl = model.run_with_hooks(tokens, fwd_hooks=hooks)
        prob = float(jax.nn.softmax(logits_abl[-1])[target_token])
        probs_ablated.append(prob)

    return base_prob, probs_ablated

# Factual recall
fact_base, fact_ablated = ablate_mlp_layers("The capital of France is", " Paris")

# Syntactic prediction
synt_base, synt_ablated = ablate_mlp_layers("The cat sat on the", " mat")

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
layers = range(model.cfg.n_layers)

ax1.bar(layers, fact_ablated, color='#e74c3c', alpha=0.8)
ax1.axhline(y=fact_base, color='black', linestyle='--', linewidth=1.5, label=f'Baseline P={fact_base:.3f}')
ax1.set_xlabel("Ablated MLP Layer")
ax1.set_ylabel("P(' Paris')")
ax1.set_title("Factual: 'The capital of France is' \u2192 ' Paris'")
ax1.set_xticks(list(layers))
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2.bar(layers, synt_ablated, color='#3498db', alpha=0.8)
ax2.axhline(y=synt_base, color='black', linestyle='--', linewidth=1.5, label=f'Baseline P={synt_base:.3f}')
ax2.set_xlabel("Ablated MLP Layer")
ax2.set_ylabel("P(' mat')")
ax2.set_title("Syntactic: 'The cat sat on the' \u2192 ' mat'")
ax2.set_xticks(list(layers))
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.suptitle("MLP Layer Ablation: Factual vs Syntactic Predictions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print the biggest drops
print("Layers whose ablation hurts factual recall most:")
drops = [(layer, fact_base - prob) for layer, prob in enumerate(fact_ablated)]
drops.sort(key=lambda x: x[1], reverse=True)
for layer, drop in drops[:5]:
    print(f"  Layer {layer}: P drops by {drop:.4f} (to {fact_ablated[layer]:.4f})")

## Interpreting the Ablation Results

Factual recall typically depends heavily on a few specific MLP layers — often in the middle-to-late range (around layers 8–11 for GPT-2 Small). These are the layers that "know" facts like "the capital of France is Paris." Ablating them causes a sharp drop in P(Paris).

Syntactic predictions like "The cat sat on the → mat" tend to be more distributed. No single MLP layer is solely responsible, because syntactic knowledge is spread more evenly. This aligns with the idea that factual knowledge is **localized** in specific MLP layers, while syntactic competence is a **distributed** property of the full network.

## Dead Neurons and Capacity

Not every neuron pulls its weight. A "dead" neuron is one that never fires (its post-GELU activation is always zero) across a wide range of inputs. Dead neurons are wasted model capacity — they consume parameters but contribute nothing. The fraction of dead neurons tells us how efficiently the model uses its MLP layers.

In [ ]:
# Dead neuron analysis across all layers
# Use our diverse test sequences
dead_fractions = []
total_dead = 0
total_neurons = 0

print("Dead neuron fraction by layer:")
print("=" * 45)

for layer in range(model.cfg.n_layers):
    frac, is_dead = dead_neuron_fraction(model, test_sequences, layer)
    dead_fractions.append(frac)
    n_dead = int(np.sum(is_dead))
    total_dead += n_dead
    total_neurons += len(is_dead)
    print(f"  Layer {layer:2d}: {frac:5.1%} dead ({n_dead:4d} / {len(is_dead)} neurons)")

print(f"\nOverall: {total_dead}/{total_neurons} neurons dead ({total_dead/total_neurons:.1%})")

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(model.cfg.n_layers), [f * 100 for f in dead_fractions], color='#95a5a6')
ax.set_xlabel("Layer")
ax.set_ylabel("Dead Neurons (%)")
ax.set_title("Fraction of Dead Neurons by Layer\n(neurons that never fire across diverse inputs)")
ax.set_xticks(range(model.cfg.n_layers))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## The Polysemanticity Problem

Here's the uncomfortable truth about neuron-level interpretability: **most neurons are polysemantic.** They respond to multiple unrelated inputs.

Why? Because the model has far more *features* (concepts it wants to represent) than it has neurons. GPT-2 Small has 3,072 neurons per layer, but it needs to represent tens of thousands of concepts — individual words, syntactic patterns, factual associations, semantic categories, and more. The solution: **superposition**. The model encodes many features in overlapping, nearly-orthogonal directions across its neurons. A single neuron participates in representing multiple features simultaneously.

This means that when we find a neuron that promotes geography-related tokens, that *same* neuron might also fire on dates, proper nouns, or completely unrelated patterns. It's not that the neuron "means" geography — it's a tangled mix of overlapping feature directions.

This is the fundamental limitation of neuron-level analysis, and it motivates the search for better feature decompositions. **Sparse autoencoders** (covered in G06) attempt to untangle these superposed features into clean, monosemantic directions.

In [ ]:
# Demonstrating polysemanticity:
# A neuron important for factual recall also fires on unrelated inputs

# Expand our test set with more diverse content
diverse_texts = test_texts + [
    "The quick brown fox jumps over the lazy dog near the fence",
    "Add three tablespoons of butter and stir until melted completely",
    "The function returns a list of integers sorted in ascending order",
    "She scored the winning goal in the final minute of the match",
    "The temperature dropped to minus ten degrees during the night",
    "He opened the door and walked into the dark room quietly",
    "The company reported record profits for the third quarter this year",
    "Water boils at one hundred degrees Celsius at sea level pressure",
    "The painting was created by an unknown artist in the sixteenth century",
    "Please submit your application before the deadline on Friday afternoon",
]
diverse_sequences = [model.to_tokens(t) for t in diverse_texts]

# Pick a neuron that was important for the Paris prediction
if interesting_neurons:
    poly_layer, poly_neuron, _ = sorted(
        interesting_neurons, key=lambda x: abs(x[2]), reverse=True
    )[0]
else:
    poly_layer, poly_neuron = 9, 0

# What does this neuron promote in the vocabulary?
promoted, suppressed = neuron_to_logit(model, poly_layer, poly_neuron, k=8)
print(f"Neuron L{poly_layer}N{poly_neuron} — important for predicting ' Paris'")
print(f"\nThis neuron promotes: {', '.join(tokenizer.decode([tid]) for tid, _ in promoted[:8])}")
print(f"This neuron suppresses: {', '.join(tokenizer.decode([tid]) for tid, _ in suppressed[:5])}")

# Now show it fires on unrelated things too
results = top_activating_tokens(model, diverse_sequences, poly_layer, poly_neuron, k=15)

print(f"\nTop activating tokens across diverse text:")
print("-" * 65)
for prompt_idx, pos, activation in results:
    tok_id = int(diverse_sequences[prompt_idx][pos])
    tok_str = tokenizer.decode([tok_id])
    seq = diverse_sequences[prompt_idx]
    start = max(0, pos - 3)
    end = min(len(seq), pos + 2)
    context = tokenizer.decode(np.array(seq[start:end]))
    print(f"  act={activation:+6.2f}  token={tok_str!r:12s}  context: ...{context}...")

print(f"\nNotice: this neuron fires on diverse, potentially unrelated contexts.")
print(f"This is polysemanticity — the neuron encodes multiple overlapping features.")

In [ ]:
# Visualizing neuron activation distributions
# Some neurons show clean bimodal (on/off) distributions; others are more continuous

# Pick a few neurons to compare: the polysemantic one, plus a couple from different layers
neurons_to_plot = [(poly_layer, poly_neuron)]

# Add neurons from early and late layers with high max activation
for check_layer in [0, 5, 11]:
    all_acts = []
    for seq in diverse_sequences[:6]:
        _, c = model.run_with_cache(seq)
        all_acts.append(get_neuron_activations(c, check_layer))
    stacked = np.concatenate(all_acts, axis=0)  # [total_tokens, d_mlp]
    max_per_neuron = np.max(np.abs(stacked), axis=0)
    best_neuron = int(np.argmax(max_per_neuron))
    if (check_layer, best_neuron) not in neurons_to_plot:
        neurons_to_plot.append((check_layer, best_neuron))

# Limit to 4 for a 2x2 grid
neurons_to_plot = neurons_to_plot[:4]

# Collect activations for each neuron across all diverse sequences
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (layer, neuron) in zip(axes.flat, neurons_to_plot):
    all_activations = []
    for seq in diverse_sequences:
        _, c = model.run_with_cache(seq)
        acts = get_neuron_activations(c, layer)
        all_activations.extend(acts[:, neuron].tolist())

    all_activations = np.array(all_activations)
    n_zero = np.sum(np.abs(all_activations) < 0.01)
    frac_zero = n_zero / len(all_activations)

    ax.hist(all_activations, bins=50, color='#2ecc71', alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.set_title(f"L{layer}N{neuron} ({frac_zero:.0%} near-zero)", fontsize=11)
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Count")
    ax.axvline(x=0, color='red', linewidth=0.8, linestyle='--')

plt.suptitle("Neuron Activation Distributions Across Diverse Inputs", fontsize=13)
plt.tight_layout()
plt.show()

print("Bimodal distributions (large spike at 0 + a spread of positive values) indicate")
print("a neuron that is clearly 'on' or 'off'. More continuous distributions suggest the")
print("neuron encodes graded information rather than a binary feature.")

## Key Takeaways

1. **MLPs act as key-value memories.** `W_in` rows are keys (detectors): they determine when a neuron fires. `W_out` columns are values (outputs): they determine what the neuron writes to the residual stream.

2. **`neuron_to_logit` reveals semantic roles.** Projecting a neuron's output through the unembedding shows which vocabulary items it promotes or suppresses.

3. **Specific MLP layers store specific knowledge.** Ablation shows that factual recall depends on a few critical layers, while syntactic predictions are more distributed.

4. **Dead neurons represent wasted capacity.** Some neurons never fire and contribute nothing to the model's computation.

5. **Polysemanticity is pervasive.** Neurons respond to multiple unrelated features due to superposition. This fundamental limitation of neuron-level analysis motivates sparse autoencoders (G06).

## Exercises

1. **Category neurons:** Use `neuron_to_logit` to find neurons that specifically promote colors, numbers, or names. Scan across layers and look for neurons whose top promoted tokens all belong to one category.

2. **Dead neuron stability:** Compare `dead_neuron_fraction` results across different sets of prompts. Does the set of dead neurons change with different input distributions, or is it stable?

3. **Monosemantic hunt:** Try to find a genuinely monosemantic neuron — one that fires only on one type of input. Use `top_activating_tokens` on a large diverse corpus. How rare are monosemantic neurons?

4. **Function words vs content words:** Run the layer ablation experiment on prompts that require predicting function words ("the", "is", "of") vs content words ("Paris", "Einstein"). Which MLP layers are most important for each?

## Further Reading

- Geva et al. 2021, ["Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — the foundational paper on MLPs as key-value stores.
- Bills et al. 2023, ["Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — using language models to automatically interpret individual neurons.
- Bricken et al. 2023, ["Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — using sparse autoencoders to find monosemantic features inside polysemantic neurons.
- **Next:** G06 — Sparse Autoencoders: Discovering Features.